> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# VIGOR 2.12.1 synthetic — canonical xarray/Zarr v3 data model
Executed tutorial using the complex mechanistic simulator.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
sim=vigor.simulate_genomic_dataset(n_samples=320,n_variants=1000,n_traits=8,seed=SEED,genotype_kwargs={"n_chromosomes":5,"generations":100},trait_kwargs=cfg)
X=sim.genotypes.to_pandas(); Y=sim.phenotypes.to_pandas()
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False

ds=xr.Dataset({"GT":(("sample","variant"),X.to_numpy(np.float32))},coords={"sample":X.index.astype(str),"variant":X.columns.astype(str),"chromosome":("variant",vigor.result_table(sim, "variant_info")["chrom"].astype(str).to_numpy()),"position":("variant",vigor.result_table(sim, "variant_info")["pos"].to_numpy())})
print(ds)
print("logical GT MB",ds.GT.nbytes/2**20,"packed2 theoretical MB",ds.GT.size*2/8/2**20)


For production, BED/PGEN remain packed read backends; dense derived arrays and optional packed hard-call materializations are written to Zarr v3.

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as td:
    path=Path(td)/"genotypes.zarr"
    vigor.write_genotype_zarr(ds,path)
    restored=vigor.open_genotype_dataset(path,source_format="zarr")
    assert dict(restored.sizes)==dict(ds.sizes)
    print("Zarr v3 genotype round trip:",dict(restored.sizes))
